<a href="https://colab.research.google.com/github/Anonymous-Chris/recursive-llm-degradation-research/blob/domain-evaluation/src/notebooks/combined_evaluation_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recursive LLM Degradation — Combined Evaluation

This notebook is the **evaluation-only** pipeline for the recursive generation experiment.

It combines the two evaluation parts assigned to this work:

1. **Domain-specific medical evaluation**
   - Disease entity retention
   - Chemical entity retention
   - Entity precision / recall / F1
   - Entity frequency
   - Reference-based entity mismatch rate

2. **Model / behavioral evaluation**
   - Answer length
   - 3-gram repetition rate
   - Conditional answer-only perplexity

The same fixed 1,000 PubMedQA labeled examples are used as the gold evaluation set for every generation.

**Important:** the reference-based entity mismatch rate is a mismatch-with-gold proxy. It is not a definitive factual hallucination detector.

## 1. Environment and project paths

In [4]:
from google.colab import files
import os
import zipfile
import shutil
from pathlib import Path

# Upload repository ZIP
uploaded = files.upload()

zip_name = next(iter(uploaded.keys()))
zip_path = Path("/content") / zip_name

print("Uploaded:", zip_path)

# Extract ZIP
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content")

# Find the extracted repository root
repo_candidates = []

for p in Path("/content").iterdir():
    if p.is_dir() and p.name.startswith("recursive-llm-degradation-research"):
        if (p / "requirements.txt").exists() and (p / "src").exists():
            repo_candidates.append(p)

if not repo_candidates:
    raise FileNotFoundError(
        "Could not find extracted repository containing requirements.txt and src/"
    )

PROJECT_DIR = repo_candidates[0]

print("Repository found at:")
print(PROJECT_DIR)

print("\nRepository contents:")
print([p.name for p in PROJECT_DIR.iterdir()])

Saving recursive-llm-degradation-research-domain-evaluation.zip to recursive-llm-degradation-research-domain-evaluation (1).zip
Uploaded: /content/recursive-llm-degradation-research-domain-evaluation (1).zip
Repository found at:
/content/recursive-llm-degradation-research-domain-evaluation

Repository contents:
['.gitignore', 'plan of action.txt', 'README.md', 'requirements.txt', 'src']


In [9]:
from pathlib import Path
import math
import re
import unicodedata
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)

# ============================================================
# CHECK HARDWARE
# ============================================================

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# FIND THE EXTRACTED REPOSITORY
# ============================================================

CONTENT_DIR = Path("/content")

repo_candidates = []

for p in CONTENT_DIR.iterdir():
    if not p.is_dir():
        continue

    if (
        p.name.startswith("recursive-llm-degradation-research")
        and (p / "requirements.txt").exists()
        and (p / "src").exists()
    ):
        repo_candidates.append(p)


if not repo_candidates:
    raise FileNotFoundError(
        "Repository not found in /content.\n\n"
        "Please upload the ZIP of the domain-evaluation branch, "
        "extract it, and run this cell again."
    )


PROJECT_DIR = repo_candidates[0]


# ============================================================
# PROJECT PATHS
# ============================================================

RESULTS_DIR = PROJECT_DIR / "results"
PREDICTION_DIR = RESULTS_DIR / "predictions"
EVALUATION_DIR = RESULTS_DIR / "evaluation"

EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
NER_MODEL_NAME = "HFpf/pubmedbert-bc5cdr-ner"

GENERATIONS = ["G0", "G1", "G2", "G3"]

PREDICTION_COLUMN = "predicted_answer"


# ============================================================
# DISPLAY
# ============================================================

print("\nProject:", PROJECT_DIR)
print("Prediction directory:", PREDICTION_DIR)
print("Evaluation directory:", EVALUATION_DIR)
print("Generations:", GENERATIONS)

print("\nRepository contents:")
for item in sorted(PROJECT_DIR.iterdir()):
    print(" -", item.name)

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

Project: /content/recursive-llm-degradation-research-domain-evaluation
Prediction directory: /content/recursive-llm-degradation-research-domain-evaluation/results/predictions
Evaluation directory: /content/recursive-llm-degradation-research-domain-evaluation/results/evaluation
Generations: ['G0', 'G1', 'G2', 'G3']

Repository contents:
 - .gitignore
 - README.md
 - plan of action.txt
 - requirements.txt
 - results
 - src


## 2. Load the fixed 1,000-example gold evaluation set

In [10]:
labeled_data = load_dataset(
    "qiaojin/PubMedQA",
    "pqa_labeled",
    split="train",
)

gold_df = pd.DataFrame(labeled_data)
gold_df = gold_df.rename(columns={"long_answer": "gold_answer"})
gold_df["pubid"] = gold_df["pubid"].astype(str)

required_gold_columns = {"pubid", "question", "context", "gold_answer"}
missing_gold = required_gold_columns - set(gold_df.columns)
if missing_gold:
    raise ValueError(f"Missing gold columns: {sorted(missing_gold)}")

if len(gold_df) != 1000:
    raise ValueError(f"Expected 1,000 labeled examples, found {len(gold_df)}.")

if gold_df["pubid"].duplicated().any():
    raise ValueError("Gold evaluation set contains duplicate pubids.")

print("Gold evaluation rows:", len(gold_df))
print("Unique pubids:", gold_df["pubid"].nunique())
display(gold_df[["pubid", "question", "gold_answer"]].head(3))

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Gold evaluation rows: 1000
Unique pubids: 1000


,pubid,question,gold_answer
0,21645374,Do mitochondria play a role in remodelling lac...,Results depicted mitochondrial dynamics in viv...
1,16418930,Landolt C and snellen e acuity: differences in...,"Using the charts described, there was only a s..."
2,9488747,"Syncope during bathing in infants, a pediatric...","""Aquagenic maladies"" could be a pediatric form..."


## 3. Load generation prediction CSVs

In [13]:
prediction_tables = {}

for generation in GENERATIONS:
    generation_lower = generation.lower()

    path = PREDICTION_DIR / (
        f"{MODEL_NAME.replace('/', '-')}-{generation_lower}-predictions.csv"
    )

    if not path.exists():
        print(f"{generation}: NOT FOUND -> {path}")
        continue

    pred_df = pd.read_csv(path)

    if PREDICTION_COLUMN not in pred_df.columns:
        raise ValueError(
            f"{generation}: expected column '{PREDICTION_COLUMN}', "
            f"found {pred_df.columns.tolist()}"
        )

    pred_df["pubid"] = pred_df["pubid"].astype(str)

    if pred_df["pubid"].duplicated().any():
        raise ValueError(
            f"{generation}: duplicate pubids in prediction CSV."
        )

    if len(pred_df) != 1000:
        raise ValueError(
            f"{generation}: expected 1,000 predictions, "
            f"found {len(pred_df)}."
        )

    merged = gold_df[
        ["pubid", "question", "context", "gold_answer"]
    ].merge(
        pred_df[["pubid", PREDICTION_COLUMN]],
        on="pubid",
        how="left",
        validate="one_to_one",
    )

    missing_predictions = merged[PREDICTION_COLUMN].isna().sum()

    if missing_predictions:
        raise ValueError(
            f"{generation}: {missing_predictions} gold examples "
            f"have no prediction."
        )

    prediction_tables[generation] = merged

    print(f"{generation}: loaded {len(merged)} aligned predictions")


# TEST-FRIENDLY BEHAVIOR:
# Do not crash if prediction CSVs have not been generated yet.

if not prediction_tables:
    print("\nNo prediction CSVs are available yet.")
    print("The evaluation pipeline is ready, but G0-G3 predictions")
    print("must be generated by the recursive-generation notebook first.")
else:
    print("\nAvailable generations:", list(prediction_tables))

G0: NOT FOUND -> /content/recursive-llm-degradation-research-domain-evaluation/results/predictions/Qwen-Qwen2.5-0.5B-g0-predictions.csv
G1: NOT FOUND -> /content/recursive-llm-degradation-research-domain-evaluation/results/predictions/Qwen-Qwen2.5-0.5B-g1-predictions.csv
G2: NOT FOUND -> /content/recursive-llm-degradation-research-domain-evaluation/results/predictions/Qwen-Qwen2.5-0.5B-g2-predictions.csv
G3: NOT FOUND -> /content/recursive-llm-degradation-research-domain-evaluation/results/predictions/Qwen-Qwen2.5-0.5B-g3-predictions.csv

No prediction CSVs are available yet.
The evaluation pipeline is ready, but G0-G3 predictions
must be generated by the recursive-generation notebook first.


## 4. Medical NER setup

PubMedBERT BC5CDR is used **only for evaluation**, not for QLoRA training.

The model provides the Disease and Chemical categories used by the domain-specific evaluation.

In [14]:
NER_DEVICE = 0 if torch.cuda.is_available() else -1

pubmed_ner_pipeline = pipeline(
    "token-classification",
    model=NER_MODEL_NAME,
    aggregation_strategy="simple",
    device=NER_DEVICE,
    stride=128,
)

print("NER model:", NER_MODEL_NAME)
print("NER device:", "cuda:0" if NER_DEVICE == 0 else "cpu")
print("Tokenizer max length:", pubmed_ner_pipeline.tokenizer.model_max_length)

config.json:   0%|          | 0.00/918 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/379 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/679k [00:00<?, ?B/s]

NER model: HFpf/pubmedbert-bc5cdr-ner
NER device: cuda:0
Tokenizer max length: 1000000000000000019884624838656


## 5. NER and entity-matching helpers

In [16]:
LABEL_MAP = {
    "LABEL_0": ("O", None),
    "LABEL_1": ("B", "Chemical"),
    "LABEL_2": ("B", "Disease"),
    "LABEL_3": ("I", "Disease"),
    "LABEL_4": ("I", "Chemical"),
}


def label_to_entity_type(label):
    # Some versions of the pipeline may already return the human-readable type.
    if label in {"Disease", "Chemical"}:
        return "B", label

    return LABEL_MAP.get(label, (None, None))


def merge_pubmed_predictions(text, predictions):
    """
    Convert PubMedBERT token-level predictions into entity spans.
    Works with either LABEL_* outputs or Disease/Chemical labels.
    """
    pieces = []

    for pred in predictions:
        label = pred.get("entity_group", pred.get("entity"))
        prefix, entity_type = label_to_entity_type(label)

        if entity_type is None:
            continue

        start = pred.get("start")
        end = pred.get("end")

        if start is None or end is None:
            continue

        pieces.append({
            "prefix": prefix,
            "type": entity_type,
            "start": int(start),
            "end": int(end),
            "score": float(pred.get("score", 0.0)),
        })

    pieces.sort(key=lambda x: (x["start"], x["end"]))

    merged = []

    for piece in pieces:
        if piece["prefix"] == "B":
            merged.append({
                "text": text[piece["start"]:piece["end"]],
                "type": piece["type"],
                "start": piece["start"],
                "end": piece["end"],
                "score": piece["score"],
            })
            continue

        if (
            merged
            and merged[-1]["type"] == piece["type"]
            and piece["start"] <= merged[-1]["end"] + 2
        ):
            merged[-1]["end"] = piece["end"]
            merged[-1]["text"] = text[
                merged[-1]["start"]:merged[-1]["end"]
            ]
            merged[-1]["score"] = (
                merged[-1]["score"] + piece["score"]
            ) / 2.0
        else:
            merged.append({
                "text": text[piece["start"]:piece["end"]],
                "type": piece["type"],
                "start": piece["start"],
                "end": piece["end"],
                "score": piece["score"],
            })

    return merged


def normalize_entity_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("##", "")
    text = re.sub(r"\s+", " ", text).strip(" \t\n.,;:()[]{}\"'")
    return text.lower()


def deduplicate_entities(entities):
    cleaned = []
    seen = set()

    for entity in entities:
        normalized = normalize_entity_text(entity["text"])
        if not normalized:
            continue

        key = (
            entity["type"],
            entity["start"],
            entity["end"],
            normalized,
        )

        if key in seen:
            continue

        seen.add(key)

        cleaned.append({
            "text": entity["text"].strip(),
            "normalized_text": normalized,
            "type": entity["type"],
            "start": entity["start"],
            "end": entity["end"],
            "score": float(entity["score"]),
        })

    return cleaned


def run_ner(texts, batch_size=8, description="NER"):
    all_predictions = []

    texts = ["" if pd.isna(x) else str(x) for x in texts]

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc=description,
    ):
        batch = texts[start:start + batch_size]
        batch_predictions = pubmed_ner_pipeline(
            batch,
            batch_size=batch_size,
        )
        all_predictions.extend(batch_predictions)

    entities = [
        deduplicate_entities(
            merge_pubmed_predictions(text, predictions)
        )
        for text, predictions in zip(texts, all_predictions)
    ]

    return entities


def get_entity_set(entities, entity_type):
    return {
        entity["normalized_text"]
        for entity in entities
        if entity["type"] == entity_type
        and entity["normalized_text"]
    }


def calculate_sample_metrics(gold_entities, pred_entities, entity_type):
    gold_set = get_entity_set(gold_entities, entity_type)
    pred_set = get_entity_set(pred_entities, entity_type)

    matched = gold_set & pred_set

    gold_count = len(gold_set)
    pred_count = len(pred_set)
    matched_count = len(matched)

    recall = matched_count / gold_count if gold_count else np.nan
    precision = matched_count / pred_count if pred_count else np.nan

    if np.isfinite(precision) and np.isfinite(recall) and precision + recall:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = np.nan

    return {
        "gold_count": gold_count,
        "pred_count": pred_count,
        "matched_count": matched_count,
        "retention": recall,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def micro_metrics(df, prefix):
    gold_disease = df["gold_disease_count"].sum()
    pred_disease = df[f"{prefix}_disease_count"].sum()
    matched_disease = df[f"{prefix}_matched_disease_count"].sum()

    disease_precision = (
        matched_disease / pred_disease if pred_disease else 0.0
    )
    disease_recall = (
        matched_disease / gold_disease if gold_disease else 0.0
    )
    disease_f1 = (
        2 * disease_precision * disease_recall
        / (disease_precision + disease_recall)
        if disease_precision + disease_recall
        else 0.0
    )

    gold_chemical = df["gold_chemical_count"].sum()
    pred_chemical = df[f"{prefix}_chemical_count"].sum()
    matched_chemical = df[f"{prefix}_matched_chemical_count"].sum()

    chemical_precision = (
        matched_chemical / pred_chemical if pred_chemical else 0.0
    )
    chemical_recall = (
        matched_chemical / gold_chemical if gold_chemical else 0.0
    )
    chemical_f1 = (
        2 * chemical_precision * chemical_recall
        / (chemical_precision + chemical_recall)
        if chemical_precision + chemical_recall
        else 0.0
    )

    return {
        "disease_gold": gold_disease,
        "disease_predicted": pred_disease,
        "disease_matched": matched_disease,
        "disease_precision": disease_precision,
        "disease_recall": disease_recall,
        "disease_f1": disease_f1,
        "chemical_gold": gold_chemical,
        "chemical_predicted": pred_chemical,
        "chemical_matched": matched_chemical,
        "chemical_precision": chemical_precision,
        "chemical_recall": chemical_recall,
        "chemical_f1": chemical_f1,
    }

## 6. Compute gold medical entities once

In [17]:
gold_entities_clean = run_ner(
    gold_df["gold_answer"].tolist(),
    batch_size=8,
    description="Gold NER",
)

print("Gold entity sets:", len(gold_entities_clean))

# Quick inspection
for i in range(min(3, len(gold_entities_clean))):
    print("=" * 70)
    print(f"Example {i + 1}")
    for entity in gold_entities_clean[i]:
        print(f"{entity['text']} -> {entity['type']}")

Gold NER:   0%|          | 0/125 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Gold entity sets: 1000
Example 1
CsA -> Chemical
Example 2
strabismus amblyopia -> Disease
Example 3
Aquagenic maladies -> Disease
aquagenic urticaria -> Disease


## 7. Behavioral metrics and fixed Qwen evaluator

In [18]:
eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
eval_tokenizer.padding_side = "left"

if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

eval_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
)
eval_model.eval()

print("Fixed evaluator loaded:", MODEL_NAME)
print("Device:", eval_model.device)


def context_to_string(context):
    if isinstance(context, dict):
        contexts = context.get("contexts")
        if isinstance(contexts, (list, tuple)):
            return " ".join(str(x) for x in contexts)
        if contexts is not None:
            return str(contexts)
        return str(context)

    if isinstance(context, (list, tuple)):
        return " ".join(str(x) for x in context)

    return str(context)


def build_prompt(row):
    return (
        f"### Question:\n{row['question']}\n\n"
        f"### Context:\n{context_to_string(row['context'])}\n\n"
        "### Long Answer:\n"
    )


def word_count(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    return len(text.split())


def token_count(text):
    if not isinstance(text, str) or not text.strip():
        return 0

    return len(
        eval_tokenizer(
            text,
            add_special_tokens=False,
        )["input_ids"]
    )


def repetition_rate(text, n=3):
    if not isinstance(text, str) or not text.strip():
        return 0.0

    words = re.findall(r"\b\w+\b", text.lower())

    if len(words) < n:
        return 0.0

    ngrams = [
        tuple(words[i:i+n])
        for i in range(len(words) - n + 1)
    ]

    return 1.0 - (len(set(ngrams)) / len(ngrams))


def calculate_batch_perplexity(rows, answer_column):
    input_id_lists = []
    label_id_lists = []

    for row in rows:
        prompt_ids = eval_tokenizer(
            build_prompt(row),
            add_special_tokens=False,
            truncation=True,
            max_length=1024,
        )["input_ids"]

        answer_ids = eval_tokenizer(
            str(row[answer_column]),
            add_special_tokens=False,
        )["input_ids"]

        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids

        input_id_lists.append(input_ids)
        label_id_lists.append(labels)

    batch = eval_tokenizer.pad(
        [{"input_ids": ids} for ids in input_id_lists],
        padding=True,
        return_tensors="pt",
    )

    input_ids = batch["input_ids"].to(eval_model.device)
    attention_mask = batch["attention_mask"].to(eval_model.device)

    max_len = input_ids.shape[1]
    labels = torch.full(
        (len(label_id_lists), max_len),
        -100,
        dtype=torch.long,
    )

    for i, label_ids in enumerate(label_id_lists):
        pad_length = max_len - len(label_ids)
        labels[i, pad_length:] = torch.tensor(
            label_ids,
            dtype=torch.long,
        )

    labels = labels.to(eval_model.device)

    with torch.no_grad():
        logits = eval_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).logits

    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    loss_per_token = torch.nn.functional.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        reduction="none",
        ignore_index=-100,
    ).view(shift_labels.shape)

    perplexities = []

    for i in range(len(rows)):
        valid_tokens = shift_labels[i] != -100

        if not valid_tokens.any():
            perplexities.append(np.nan)
            continue

        mean_loss = loss_per_token[i][valid_tokens].mean().item()
        perplexities.append(math.exp(mean_loss))

    return perplexities

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Fixed evaluator loaded: Qwen/Qwen2.5-0.5B
Device: cuda:0


## 8. Reusable evaluator for G0 / G1 / G2 / G3

In [19]:
def evaluate_generation(
    eval_df,
    prediction_column,
    generation_name,
    gold_entities,
    ner_batch_size=8,
    ppl_batch_size=4,
):
    """
    Evaluate one generation on the same fixed gold evaluation set.

    Returns:
        result_df   : per-sample metrics
        summary_df  : one-row generation summary
    """
    short_name = generation_name.lower()

    texts = (
        eval_df[prediction_column]
        .fillna("")
        .astype(str)
        .tolist()
    )

    # -----------------------------
    # A. Medical NER
    # -----------------------------
    pred_entities = run_ner(
        texts,
        batch_size=ner_batch_size,
        description=f"{generation_name} NER",
    )

    domain_rows = []

    for i, (gold_entities_i, pred_entities_i) in enumerate(
        zip(gold_entities, pred_entities)
    ):
        disease = calculate_sample_metrics(
            gold_entities_i,
            pred_entities_i,
            "Disease",
        )

        chemical = calculate_sample_metrics(
            gold_entities_i,
            pred_entities_i,
            "Chemical",
        )

        domain_rows.append({
            "pubid": eval_df.iloc[i]["pubid"],

            "gold_disease_count": disease["gold_count"],
            f"{short_name}_disease_count": disease["pred_count"],
            f"{short_name}_matched_disease_count": disease["matched_count"],
            f"{short_name}_disease_retention": disease["retention"],
            f"{short_name}_disease_precision": disease["precision"],
            f"{short_name}_disease_recall": disease["recall"],
            f"{short_name}_disease_f1": disease["f1"],

            "gold_chemical_count": chemical["gold_count"],
            f"{short_name}_chemical_count": chemical["pred_count"],
            f"{short_name}_matched_chemical_count": chemical["matched_count"],
            f"{short_name}_chemical_retention": chemical["retention"],
            f"{short_name}_chemical_precision": chemical["precision"],
            f"{short_name}_chemical_recall": chemical["recall"],
            f"{short_name}_chemical_f1": chemical["f1"],
        })

    result_df = pd.DataFrame(domain_rows)

    # -----------------------------
    # B. Behavioral metrics
    # -----------------------------
    behavioral = pd.DataFrame({
        "pubid": eval_df["pubid"],
        f"{short_name}_word_count": [word_count(x) for x in texts],
        f"{short_name}_token_count": [token_count(x) for x in texts],
        f"{short_name}_repetition_rate": [
            repetition_rate(x) for x in texts
        ],
    })

    result_df = result_df.merge(
        behavioral,
        on="pubid",
        how="left",
        validate="one_to_one",
    )

    # -----------------------------
    # C. Conditional PPL
    # -----------------------------
    rows = eval_df.to_dict("records")
    perplexities = []

    for start in tqdm(
        range(0, len(rows), ppl_batch_size),
        desc=f"{generation_name} perplexity",
    ):
        batch_rows = rows[start:start + ppl_batch_size]
        perplexities.extend(
            calculate_batch_perplexity(
                batch_rows,
                prediction_column,
            )
        )

    result_df[f"{short_name}_perplexity"] = perplexities

    # -----------------------------
    # D. Reference-based mismatch
    # -----------------------------
    pred_disease_total = result_df[
        f"{short_name}_disease_count"
    ].sum()

    matched_disease_total = result_df[
        f"{short_name}_matched_disease_count"
    ].sum()

    pred_chemical_total = result_df[
        f"{short_name}_chemical_count"
    ].sum()

    matched_chemical_total = result_df[
        f"{short_name}_matched_chemical_count"
    ].sum()

    disease_reference_mismatch = (
        1.0 - matched_disease_total / pred_disease_total
        if pred_disease_total > 0
        else 0.0
    )

    chemical_reference_mismatch = (
        1.0 - matched_chemical_total / pred_chemical_total
        if pred_chemical_total > 0
        else 0.0
    )

    # -----------------------------
    # E. Summary
    # -----------------------------
    micro = micro_metrics(result_df, short_name)

    summary = {
        "generation": generation_name,

        "disease_retention": micro["disease_recall"],
        "disease_precision": micro["disease_precision"],
        "disease_recall": micro["disease_recall"],
        "disease_f1": micro["disease_f1"],
        "disease_reference_mismatch_rate": disease_reference_mismatch,

        "chemical_retention": micro["chemical_recall"],
        "chemical_precision": micro["chemical_precision"],
        "chemical_recall": micro["chemical_recall"],
        "chemical_f1": micro["chemical_f1"],
        "chemical_reference_mismatch_rate": chemical_reference_mismatch,

        "gold_disease_entities_per_answer":
            result_df["gold_disease_count"].mean(),
        "predicted_disease_entities_per_answer":
            result_df[f"{short_name}_disease_count"].mean(),

        "gold_chemical_entities_per_answer":
            result_df["gold_chemical_count"].mean(),
        "predicted_chemical_entities_per_answer":
            result_df[f"{short_name}_chemical_count"].mean(),

        "avg_words":
            result_df[f"{short_name}_word_count"].mean(),
        "avg_tokens":
            result_df[f"{short_name}_token_count"].mean(),
        "avg_repetition_rate":
            result_df[f"{short_name}_repetition_rate"].mean(),

        "mean_perplexity":
            result_df[f"{short_name}_perplexity"].mean(),
        "median_perplexity":
            result_df[f"{short_name}_perplexity"].median(),
    }

    summary_df = pd.DataFrame([summary])

    return result_df, summary_df

## 9. Run the combined evaluation

In [21]:
all_generation_results = {}
all_generation_summaries = []

if not prediction_tables:
    print(
        "Skipping evaluation because no prediction CSVs "
        "are available yet."
    )
else:
    for generation, generation_df in prediction_tables.items():

        result_df, summary_df = evaluate_generation(
            eval_df=generation_df,
            prediction_column=PREDICTION_COLUMN,
            generation_name=generation,
            gold_entities=gold_entities_clean,
        )

        all_generation_results[generation] = result_df
        all_generation_summaries.append(summary_df)

        per_generation_path = (
            EVALUATION_DIR / f"{generation.lower()}_evaluation.csv"
        )

        result_df.to_csv(
            per_generation_path,
            index=False,
        )

        print(f"\nSaved: {per_generation_path}")

    combined_summary = pd.concat(
        all_generation_summaries,
        ignore_index=True,
    )

    display(combined_summary)

Skipping evaluation because no prediction CSVs are available yet.


## 10. Final comparison table

This is the main table to use for the domain-specific + behavioral part of the study.

The lexical and semantic metrics from the partner's evaluation can be merged into this table later using the `generation` column.

In [22]:
if not all_generation_summaries:

    print(
        "Final comparison table will be created "
        "after G0-G3 prediction CSVs are available."
    )

else:

    comparison_columns = [
        "generation",

        "disease_retention",
        "disease_precision",
        "disease_recall",
        "disease_f1",
        "disease_reference_mismatch_rate",

        "chemical_retention",
        "chemical_precision",
        "chemical_recall",
        "chemical_f1",
        "chemical_reference_mismatch_rate",

        "gold_disease_entities_per_answer",
        "predicted_disease_entities_per_answer",
        "gold_chemical_entities_per_answer",
        "predicted_chemical_entities_per_answer",

        "avg_words",
        "avg_tokens",
        "avg_repetition_rate",

        "mean_perplexity",
        "median_perplexity",
    ]

    comparison_table = combined_summary[
        comparison_columns
    ].copy()

    display(comparison_table)

    comparison_path = (
        EVALUATION_DIR
        / "combined_domain_behavioral_summary.csv"
    )

    comparison_table.to_csv(
        comparison_path,
        index=False,
    )

    print("Saved:", comparison_path)

Final comparison table will be created after G0-G3 prediction CSVs are available.


In [24]:
# Optional: combine all per-sample results across generations.
# This section is skipped until prediction CSVs are available.

if not all_generation_results:

    print(
        "Skipping combined per-sample output because "
        "no generation evaluation results are available yet."
    )

else:

    combined_per_sample = None

    for generation, result_df in all_generation_results.items():

        prefix = generation.lower()

        columns_to_keep = [
            "pubid",

            "gold_disease_count",
            "gold_chemical_count",

            f"{prefix}_disease_count",
            f"{prefix}_matched_disease_count",
            f"{prefix}_disease_retention",
            f"{prefix}_disease_precision",
            f"{prefix}_disease_recall",
            f"{prefix}_disease_f1",

            f"{prefix}_chemical_count",
            f"{prefix}_matched_chemical_count",
            f"{prefix}_chemical_retention",
            f"{prefix}_chemical_precision",
            f"{prefix}_chemical_recall",
            f"{prefix}_chemical_f1",

            f"{prefix}_word_count",
            f"{prefix}_token_count",
            f"{prefix}_repetition_rate",
            f"{prefix}_perplexity",
        ]

        current = result_df[columns_to_keep].copy()

        if combined_per_sample is None:

            combined_per_sample = current

        else:

            current = current.drop(
                columns=[
                    "gold_disease_count",
                    "gold_chemical_count",
                ]
            )

            combined_per_sample = combined_per_sample.merge(
                current,
                on="pubid",
                how="left",
                validate="one_to_one",
            )

    per_sample_path = (
        EVALUATION_DIR
        / "combined_domain_behavioral_per_sample.csv"
    )

    combined_per_sample.to_csv(
        per_sample_path,
        index=False,
    )

    print("Saved:", per_sample_path)
    display(combined_per_sample.head(3))

Skipping combined per-sample output because no generation evaluation results are available yet.


## Evaluation notes

- The fixed gold set is the same 1,000 PubMedQA labeled examples for every generation.
- Prediction CSVs are aligned by `pubid`, not by row order.
- PubMedBERT BC5CDR is used only for Disease/Chemical entity extraction during evaluation.
- Disease/Chemical retention is equivalent to entity recall under the matching rule used here.
- Reference-based entity mismatch is a gold-reference mismatch proxy, not a definitive factual hallucination rate.
- Perplexity is conditional answer-only perplexity under the fixed `Qwen/Qwen2.5-0.5B` evaluator.
- The evaluation notebook does not train or generate answers; it only consumes the prediction CSVs produced by the recursive-generation pipeline.